# 82 Colocation x Veeva expert signal test on KER-10 writers (Phase 2, R5)

Re-run of `60_veeva_colocation_idn_combined_signal_test` asked by R5 in `80_` (P2-D5): the writer definition changes from LRx V7 weekly `NBRX > 0` to **KER-10 `active_nbrx_writers`** in `kerendia_metric_views.mv_kerendia_nbrx` (paid NBRx, IQVIA LAAD sub-national). Everything else is kept from `60_`: universe (294,727 providers), practice / IDN keys, windows (look-back 2025-07-25 to 2025-10-23, outcome 2025-10-24 to 2026-10-23), self exclusion, new-writer view (no Kerendia LRx TRx in the look-back), segments.

Both writer definitions are run side by side in the same notebook, so the `60_` numbers are reproduced (`writer_def = LRX_V7`) next to the new ones (`KER10`). Expert = best in-scope decile 8 to 10 from the `56_` T3 table (anchor 2025-10-23), which `82_` check L2 showed equals the Phase 2 snapshot T3.

Output: `OUT/features_phase2/tests/colocation_ker10/run=<ts>/` (one row per test x group x writer definition).

In [0]:
from pyspark.sql import functions as F
import pandas as pd, numpy as np, datetime as dt

C = "`ph-com-ai-us-prod-catalog-2553213839599380`"
OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
UNIVERSE_PATH = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/HCP_Universe_without_call_hcps"
LB_START, LB_END = "2025-07-25", "2025-10-23"
OUT_START, OUT_END = "2025-10-24", "2026-10-23"
SEG_TABLE = f"{C}.kerendia_cp.cp_ker_hcp_2026q4"
MV = f"{C}.kerendia_metric_views.mv_kerendia_nbrx"
RUN_TS = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")

univ = spark.read.parquet(UNIVERSE_PATH).select("PROVIDER_ID").distinct()
xw = spark.table(f"{C}.kerendia_nba_3.source_provider_bhid_npi_mapping").select("PROVIDER_ID", "BH_ID")
affl = (spark.table(f"{C}.pace_ker_in_house_schema.cust_mdm_best_affl_publish_pd_outp_static")
             .filter("BRAND_RANK = 1").select("BH_ID", F.col("BHO_ID").alias("L1_ID"), "HCP_SPECIALTY"))
hier = (spark.table(f"{C}.pace_ker_in_house_schema.mdm_stg_acct_hrchy_ker")
             .select(F.col("L1_CHILD_BHO_ID").alias("L1_ID"), "L4_IDN_BHO_ID", "L4_IDN_ORGNAME")
             .withColumn("L4_ID", F.when(F.col("L4_IDN_ORGNAME").isNull() | F.upper(F.trim("L4_IDN_ORGNAME")).isin("NA", "MULTIPLE DOCTORS"), None)
                                   .otherwise(F.col("L4_IDN_BHO_ID")))
             .select("L1_ID", "L4_ID", "L4_IDN_ORGNAME"))
seg = (spark.table(SEG_TABLE).filter(F.col("FF").isin("CR1", "CR2", "CR4", "CR5") &
                                     ((F.upper("PRODUCT_1") == "KERENDIA") | (F.upper("PRODUCT_2") == "KERENDIA")))
            .select(F.col("BAYER_ID").cast("string").alias("BH_ID"), F.col("NOTE_2").alias("SEGMENT")).dropDuplicates(["BH_ID"]))
expert = spark.read.parquet(f"{OUT}features/t3/anchor=2025-10-23").select("PROVIDER_ID", (F.coalesce("veeva_best_decile", F.lit(0)) >= 8).cast("int").alias("EXPERT"))

# writers, definition 1 = 60_ (LRx V7 weekly NBRX > 0)
lrx = spark.table(f"{C}.kerendia_laad_based_lrx.kerendia_laad_based_lrx_v7").filter("PERIOD_TYPE = 'W' AND upper(PRODUCT_GROUP) = 'KERENDIA'")
lrx_w = lambda a, b: lrx.filter("NBRX > 0").filter(F.col("PERIOD_ID").between(a, b)).select("PROVIDER_ID").distinct()
# writers, definition 2 = KER-10 active NBRx writers (metric view; aggregate per provider first, then join)
def ker10_w(a, b):
    return spark.sql(f"""SELECT `provider_id` AS PROVIDER_ID, MEASURE(`active_nbrx_writers`) AS W
                         FROM {MV} WHERE `week_ending` >= '{a}' AND `week_ending` <= '{b}' GROUP BY ALL""").filter("W > 0").select("PROVIDER_ID")
W = {"LRX_V7": (lrx_w(LB_START, LB_END), lrx_w(OUT_START, OUT_END)), "KER10": (ker10_w(LB_START, LB_END), ker10_w(OUT_START, OUT_END))}
lb_any = lrx.filter("TRX > 0").filter(F.col("PERIOD_ID").between(LB_START, LB_END)).select("PROVIDER_ID").distinct().withColumn("ANY_TRX_LB", F.lit(1))

b = (univ.join(xw, "PROVIDER_ID", "left").join(affl, "BH_ID", "left").join(hier, "L1_ID", "left").join(seg, "BH_ID", "left")
         .join(expert, "PROVIDER_ID", "left").join(lb_any, "PROVIDER_ID", "left"))
for d, (lb, ou) in W.items():
    b = b.join(lb.withColumn(f"LB_{d}", F.lit(1)), "PROVIDER_ID", "left").join(ou.withColumn(f"OUT_{d}", F.lit(1)), "PROVIDER_ID", "left")
base0 = b.fillna(0, ["EXPERT", "ANY_TRX_LB"] + [f"{p}_{d}" for d in W for p in ("LB", "OUT")]).toPandas()
base0["NEW"] = 1 - base0.ANY_TRX_LB

mx = spark.sql(f"SELECT MAX(`week_ending`) AS mx FROM (SELECT `week_ending`, MEASURE(`paid_nbrx`) AS n FROM {MV} WHERE `week_ending` >= '2026-09-01' GROUP BY ALL)").first().mx
print("rows:", len(base0), "| distinct providers:", base0.PROVIDER_ID.nunique(), "| experts:", int(base0.EXPERT.sum()), "| KER-10 last week_ending:", mx, "(outcome ends", OUT_END, ")")
agree = pd.DataFrame([{"window": w, "LRX_V7": int(base0[f"{w}_LRX_V7"].sum()), "KER10": int(base0[f"{w}_KER10"].sum()),
                       "both": int((base0[f"{w}_LRX_V7"] & base0[f"{w}_KER10"]).sum()),
                       "only_LRX_V7": int((base0[f"{w}_LRX_V7"] & ~base0[f"{w}_KER10"].astype(bool)).sum()),
                       "only_KER10": int((base0[f"{w}_KER10"] & ~base0[f"{w}_LRX_V7"].astype(bool)).sum())} for w in ("LB", "OUT")])
display(agree)

In [0]:
def add_peer(df, key, tag):
    g, has = df[key], df[key].notna()
    for col, src in [("N", None), ("W", "LB_WRITER"), ("E", "EXPERT"), ("EW", "EXW")]:
        s = pd.Series(1, index=df.index) if src is None else df[src]
        df[f"PEER_{col}_{tag}"] = np.where(has, s.groupby(g).transform("sum") - s, np.nan)   # self excluded
    return df

def rates(df, by):
    out = df.groupby(by, dropna=False, observed=True).agg(HCPS=("OUT_WRITER", "size"), RATE_ALL=("OUT_WRITER", "mean"))
    nw = df[df.NEW == 1].groupby(by, dropna=False, observed=True).agg(NEW_HCPS=("OUT_WRITER", "size"), RATE_NEW=("OUT_WRITER", "mean"))
    out = out.join(nw)
    out["RATE_ALL"], out["RATE_NEW"] = (100 * out.RATE_ALL).round(1), (100 * out.RATE_NEW).round(1)
    return out.reset_index()

size_band = lambda n: pd.cut(n, [0, 4, 19, np.inf], labels=["1-4 peers", "5-19 peers", "20+ peers"])

def run_tests(d):
    base = base0.copy()
    base["LB_WRITER"], base["OUT_WRITER"] = base[f"LB_{d}"], base[f"OUT_{d}"]
    base["EXW"] = base.EXPERT * base.LB_WRITER
    base = add_peer(add_peer(base, "L1_ID", "L1"), "L4_ID", "L4")
    T = {}
    T["0 baseline"] = rates(base.assign(ALL="all"), "ALL")
    l1p = base[base.PEER_N_L1 >= 1].copy()
    l1p["L1_PEER_EXPERT"], l1p["L1_PEER_WRITER"], l1p["SIZE"] = (l1p.PEER_E_L1 > 0).astype(int), (l1p.PEER_W_L1 > 0).astype(int), size_band(l1p.PEER_N_L1)
    T["0 L1 peer writer"] = rates(l1p, "L1_PEER_WRITER")
    T["1a L1 expert colleague"] = rates(l1p, "L1_PEER_EXPERT")
    T["1a-own"] = rates(l1p, ["EXPERT", "L1_PEER_EXPERT"])
    T["1a-size"] = rates(l1p, ["SIZE", "L1_PEER_EXPERT"])
    T["1b peer writer x expert colleague"] = rates(l1p, ["L1_PEER_WRITER", "L1_PEER_EXPERT"])
    T["1b-size"] = rates(l1p, ["SIZE", "L1_PEER_WRITER", "L1_PEER_EXPERT"])
    l4p = base[base.PEER_N_L4 >= 1].copy()
    l4p["L4_PEER_EXPERT"] = (l4p.PEER_E_L4 > 0).astype(int)
    l4p["L4_EXPERT_SHARE"], l4p["L4_WRITER_SHARE"] = l4p.PEER_E_L4 / l4p.PEER_N_L4, l4p.PEER_W_L4 / l4p.PEER_N_L4
    T["1c L4 expert colleague"] = rates(l4p, "L4_PEER_EXPERT")
    nz = l4p.L4_EXPERT_SHARE > 0
    l4p["EXPERT_SHARE_Q"] = "0 (no expert peer)"
    l4p.loc[nz, "EXPERT_SHARE_Q"] = pd.qcut(l4p.loc[nz, "L4_EXPERT_SHARE"].rank(method="first"), 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"]).astype(str)
    T["1c-share"] = rates(l4p, "EXPERT_SHARE_Q")
    l4p["TOP_WRITER_SHARE"] = (l4p.L4_WRITER_SHARE >= l4p.L4_WRITER_SHARE.quantile(0.8)).astype(int)
    l4p["TOP_EXPERT_SHARE"] = (l4p.L4_EXPERT_SHARE >= l4p.L4_EXPERT_SHARE.quantile(0.8)).astype(int)
    T["1d top writer share x top expert share"] = rates(l4p, ["TOP_WRITER_SHARE", "TOP_EXPERT_SHARE"])
    w1 = base[base.PEER_W_L1 >= 1].copy()
    w1["EXPERT_PEER_WRITER"] = (w1.PEER_EW_L1 > 0).astype(int)
    w1["N_PEER_WRITERS"] = pd.cut(w1.PEER_W_L1, [0, 1, 3, np.inf], labels=["1 writer", "2-3 writers", "4+ writers"])
    T["2a L1 expert peer writer"] = rates(w1, "EXPERT_PEER_WRITER")
    T["2a-count"] = rates(w1, ["N_PEER_WRITERS", "EXPERT_PEER_WRITER"])
    w4 = base[base.PEER_W_L4 >= 1].copy()
    w4["EXW_BAND"] = pd.cut(w4.PEER_EW_L4 / w4.PEER_W_L4, [-0.001, 0, 0.2, 0.4, 1.0], labels=["0%", "0-20%", "20-40%", "40%+"])
    w4["WRITER_SHARE_Q"] = pd.qcut((w4.PEER_W_L4 / w4.PEER_N_L4).rank(method="first"), 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
    T["2b L4 expert share of writers"] = rates(w4, "EXW_BAND")
    T["2b-ctrl"] = rates(w4, ["WRITER_SHARE_Q", "EXW_BAND"])
    ex = base[(base.EXPERT == 1) & (base.LB_WRITER == 0) & (base.PEER_N_L4 >= 1)].copy()
    ex["ADOPTION_BAND"] = pd.cut(ex.PEER_W_L4 / ex.PEER_N_L4, [-0.001, 0, 0.05, 0.10, 0.20, 1.0], labels=["0%", "0-5%", "5-10%", "10-20%", ">20%"])
    T["3a non-writing experts by IDN adoption"] = rates(ex, "ADOPTION_BAND")
    low = ex[ex.PEER_W_L4 / ex.PEER_N_L4 <= 0.05]
    T["3b low-adoption experts by segment"] = low.assign(SEG=low.SEGMENT.fillna("Non-target")).groupby("SEG").size().rename("HCPS").reset_index()
    sp = round(l4p[["L4_WRITER_SHARE", "L4_EXPERT_SHARE"]].corr(method="spearman").iloc[0, 1], 3)
    rows = []
    for t, df in T.items():
        keys = [c for c in df.columns if c not in ("HCPS", "RATE_ALL", "NEW_HCPS", "RATE_NEW")]
        for r in df.to_dict("records"):
            rows.append({"writer_def": d, "test": t, "group": " | ".join(f"{k}={r[k]}" for k in keys),
                         **{m: r.get(m) for m in ("HCPS", "RATE_ALL", "NEW_HCPS", "RATE_NEW")}})
    return pd.DataFrame(rows), sp, len(low), low.L4_ID.nunique()

res, extra = [], []
for d in W:
    r, sp, nlow, nidn = run_tests(d)
    res.append(r); extra.append({"writer_def": d, "spearman_writer_vs_expert_share": sp, "low_adoption_experts": nlow, "low_adoption_idns": nidn})
res = pd.concat(res)
display(pd.DataFrame(extra))

In [0]:
cmp = res.pivot_table(index=["test", "group"], columns="writer_def", values=["HCPS", "RATE_ALL", "RATE_NEW"], aggfunc="first", sort=False)
cmp.columns = [f"{m}_{d}" for m, d in cmp.columns]
cmp = cmp.reset_index()
for m in ("RATE_ALL", "RATE_NEW"):
    cmp[f"{m}_diff"] = (cmp[f"{m}_KER10"] - cmp[f"{m}_LRX_V7"]).round(1)
keep = ["test", "group", "HCPS_LRX_V7", "HCPS_KER10", "RATE_ALL_LRX_V7", "RATE_ALL_KER10", "RATE_ALL_diff", "RATE_NEW_LRX_V7", "RATE_NEW_KER10", "RATE_NEW_diff"]
cmp = cmp[keep]
P = f"{OUT}features_phase2/tests/colocation_ker10/run={RUN_TS}/"
spark.createDataFrame(res.astype({"HCPS": float, "RATE_ALL": float, "NEW_HCPS": float, "RATE_NEW": float})).write.mode("overwrite").parquet(P + "long")
spark.createDataFrame(cmp).write.mode("overwrite").parquet(P + "side_by_side")
print("written:", P)
display(cmp)

In [0]:
# C1 LRX_V7 reproduces 60_ base counts (assessment: 233,102 HCPs with >=1 L1 peer; 65,122 with >=1 L1 peer writer)
l1 = res[(res.writer_def == "LRX_V7") & (res.test == "0 L1 peer writer")]
print("C1 L1-peer HCPs:", int(l1.HCPS.sum()), "| with peer writer:", int(l1[l1.group == "L1_PEER_WRITER=1"].HCPS.sum()))
# C2 partitions add up per writer definition
for d in W:
    r = res[res.writer_def == d]
    print("C2", d, "| 1b cells = 1a cells:", r[r.test == "1b peer writer x expert colleague"].HCPS.sum() == r[r.test == "1a L1 expert colleague"].HCPS.sum(),
          "| baseline HCPs = universe:", int(r[r.test == "0 baseline"].HCPS.sum()) == len(base0))
# C3 tests that do not use writers in the grouping keep the same HCP counts in both definitions
same_n = cmp[cmp.test.isin(["1a L1 expert colleague", "1a-own", "1a-size", "1c L4 expert colleague", "1c-share"])]
print("C3 writer-free groupings, HCP counts equal:", bool((same_n.HCPS_LRX_V7 == same_n.HCPS_KER10).all()))